In [1]:
import xarray as xr
import pickle
import numpy as np
from datetime import datetime
import time
from tqdm import tqdm

import os
os.environ['WXSYSLIBDIR']='/g/data/gb02/mb0427/WxSysLib'
os.environ['SWTpath']='/home/565/mb0427/gdata-gb02/Australian_synoptic_weather_types/'
import sys
sys.path.append(os.environ['WXSYSLIBDIR'])
sys.path.append(os.environ['SWTpath'])

from assign.assign import assign_grid_to_SWT
from utils.general.date_utils import generate_datetimes,generate_datetimes_months
from utils.general.nci_utils import get_GADI_ERA5_filename

with open(os.environ['SWTpath']+'SWT_fields/SWT_WR_definitions.pkl', 'rb') as fp:
    SWTs = pickle.load(fp)

SWTnames = [SWTs[i]["WR"]+'-'+SWTs[i]["SWT"] for i in SWTs]

In [5]:
ds=xr.open_dataset(f"{os.environ['SWTpath']}/SWT_fields/SWT_climatology_v0.nc")
ds.close()

In [6]:
clusterSeries=ds.clusterSeries
SWTs=ds.SWT
SWTnames=np.array(SWTnames)

In [10]:
clusterSWTs=[SWTs.sel(clusterID=clusterSeries.sel(time=t)).values for t in clusterSeries.time]
clusterSWTs=np.array(clusterSWTs)

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x148edc607610>>
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-25.04/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x148edc607610>>
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-25.04/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


KeyboardInterrupt: 

In [5]:
# Create the xarray Dataset
new_ds = xr.Dataset(
    data_vars={
        "assigned_SWT": (("time"), clusterSWTs),
    },
    coords={
        "time": ds.time,
        "SWTs": SWTnames,
    }
)
new_attrs = ds.attrs
new_attrs['dataset']='Australian Synoptic Weather Types' 
new_attrs['reference']='10.22541/essoar.174250748.85665142/v1' 
new_attrs['created by']='Michael A. Barnes (ARC Centre of Excellence for 21st Century Weather, Monash University)' 
new_ds = new_ds.assign_attrs(new_attrs)

SWT_attrs={}
SWT_attrs['short_name']='SWTs' 
SWT_attrs['long_name']='Synoptic Weather Types'
SWT_attrs['description']='List of available SWTs as per Barnes et. al. (2025)'

new_ds['SWTs'] = new_ds['SWTs'].assign_attrs(SWT_attrs)

assignedSWT_attrs={}
assignedSWT_attrs['short_name']='Assigned SWT' 
assignedSWT_attrs['long_name']='Assigned SWT for each day' 

new_ds['assigned_SWT'] = new_ds['assigned_SWT'].assign_attrs(assignedSWT_attrs)

new_ds.to_netcdf(f"{os.environ['SWTpath']}/SWT_fields/SWT_climatology.nc")

In [12]:
cluster_filename=f"{os.environ['SWTpath']}/SWT_fields/SWT_data_v1.nc"
ds=xr.open_dataset(cluster_filename)
SWTs=ds.SWT
ds.close()

In [16]:
ds=xr.open_dataset(cluster_filename)
ds.close()

In [17]:
clusterSeries=ds.clusterSeries
#SWTs=ds.SWT
SWTnames=np.array(SWTnames)

In [15]:
cluster_filename=f"/g/data/if69/mb0427/SWTs/ERA5/SWT_data_reassigned_850_v1.nc"
#cluster_filename=f"{os.environ['SWTpath']}/SWT_fields/SWT_data_v1.nc"

In [18]:
clusterSWTs=[SWTs.sel(clusterID=clusterSeries.sel(time=t)).values for t in clusterSeries.time]
clusterSWTs=np.array(clusterSWTs)

In [19]:
clusterU = xr.DataArray(
        ds.clusterU.values,
        coords={"latitude": ds.latitude, 
                "longitude": ds.longitude,
                "SWT": SWTs.values,
               },
        dims=["SWT","latitude","longitude"]
    )
clusterU=clusterU.reindex(SWT=SWTnames)

clusterV = xr.DataArray(
        ds.clusterV.values,
        coords={"latitude": ds.latitude, 
                "longitude": ds.longitude,
                "SWT": SWTs.values,
               },
        dims=["SWT","latitude","longitude"]
    )
clusterV=clusterV.reindex(SWT=SWTnames)

assigned_SWTs = xr.DataArray(
        clusterSWTs,
        coords={"time": ds.time,
               },
        dims=["time"]
    )

In [20]:
new_ds = xr.Dataset(
    {'assigned_SWT': assigned_SWTs, 'clusterU': clusterU, 'clusterV': clusterV}
)

new_ds.to_netcdf(f"/g/data/if69/mb0427/SWTs/ERA5/SWT_data_reassigned_850.nc")